# FlyArystan delay case — Task 2

**Important data limitation:** no public flight-level FlyArystan operational dataset was located for this project. The supplied 1,200-flight panel is entirely simulated and clearly labelled; it is calibrated to a company-reported summer 2026 punctuality figure, but it is not a record of real flights. The tests below demonstrate the method and do not establish a real operational bottleneck or real causal effect.

The process candidate is aircraft turnaround/rotation recovery, because FlyArystan says summer schedules are intensive, turnaround times are short, and disruptions can propagate. This is a hypothesis to investigate, not a confirmed bottleneck.

Sources: FlyArystan press release (10 Sep 2026); Civil Aviation Committee official notice (Aug 2024); Tengrinews reporting of two airline-confirmed events (24 Aug and 23 Nov 2026). Details and limitations are in `data_note.pdf`.

Dependencies: `pandas`, `numpy`, `scipy`. Random seed: 20261125.

## Process map and cause screen

**Process:** previous flight lands → aircraft arrives at gate → passengers/cargo unload → aircraft is cleaned, serviced, and prepared → boarding closes → scheduled departure → actual departure delay → aircraft continues to its next rotation. The proposed waiting point is the turnaround between arrival at gate and the next departure; no real timestamps were available to locate an observed queue there.

Four distinct candidate causes are: (1) the inbound aircraft arrives late and passes delay to the next leg; (2) the planned ground-turn buffer is too short for required work; (3) adverse weather disrupts arrival or departure; and (4) an exceptional passenger/security event interrupts boarding or requires re-screening. Public reports show more than one disruption type, not their relative frequency. The synthetic panel models the first two to demonstrate the tests; it does not establish which cause is most likely in actual operations.

## 1. Regenerate the synthetic raw data

The generator uses only simulated flight labels and a stylized delay equation. It calibrates the synthetic departure-delay proxy so that 84% of rows meet a 15-minute threshold, to echo the company’s self-reported 84% summer OTP. The company’s actual KPI also counts scheduled-time arrivals, so this is not a metric-for-metric match. The coefficients are assumptions, not observed company parameters.

In [1]:
import numpy as np, pandas as pd
from pathlib import Path
SEED=20261125; N=1200
rng=np.random.default_rng(SEED)
dates=pd.date_range('2026-06-01','2026-08-31',freq='D')
flight_dates=rng.choice(dates.to_numpy(),N)
# These are synthetic route and aircraft labels, not actual FlyArystan flight assignments.
routes=np.array([f'R{i:02d}' for i in range(1,7)])
aircraft=np.array([f'SIM-A{i:02d}' for i in range(1,13)])
route=rng.choice(routes,N,p=[.24,.20,.18,.15,.13,.10])
aircraft_id=rng.choice(aircraft,N)
hour=rng.choice(np.arange(6,24),N,p=np.array([.02,.02,.02,.03,.04,.05,.07,.08,.08,.08,.08,.08,.08,.08,.07,.05,.04,.03]))
turnaround=rng.choice([35,40,45,50,55,60,70,80,90],N,p=[.08,.10,.12,.15,.16,.15,.11,.08,.05])
# Mixture calibrated as a plausible disruption input; no real flight log is used.
u=rng.random(N); inbound=np.zeros(N,dtype=float)
inbound[u<.18]=rng.uniform(5,14,(u<.18).sum())
inbound[(u>=.18)&(u<.31)]=rng.uniform(15,45,((u>=.18)&(u<.31)).sum())
inbound[u>=.31]=rng.gamma(2.0,24.0,(u>=.31).sum())
inbound=np.clip(inbound,0,150)
weather=rng.choice(['Clear','Adverse'],N,p=[.84,.16])
demand=np.clip(np.rint(rng.normal(65,17,N)),20,100).astype(int)
short=(turnaround<45).astype(int)
late_in=(inbound>=15).astype(int)
adverse=(weather=='Adverse').astype(int)
# Stylized delay-generating equation. The intercept is calibrated so exactly 84% of
# this synthetic sample meets a <=15-minute departure-delay proxy; this is NOT the
# company's OTP definition (which also counts scheduled-time arrivals).
z=rng.normal(0,9,N)+0.34*inbound+13*short+9*adverse+0.10*np.maximum(demand-65,0)
cut=np.sort(z)[int(.84*N)-1]
dep_delay=np.maximum(0,z+(15-cut))
# Apply a small continuous jitter to avoid threshold ties; deterministic from the same seed.
dep_delay=np.round(dep_delay,1)
on_time=(dep_delay<=15).astype(int)
# Raw export deliberately includes documented formatting noise, missing codes, and exact duplicate rows.
raw=pd.DataFrame({
 'record_id':np.arange(1,N+1),
 'flight_date':[d.strftime('%Y-%m-%d') if i%3==0 else d.strftime('%d %b %Y') if i%3==1 else d.strftime('%d/%m/%Y') for i,d in enumerate(pd.to_datetime(flight_dates))],
 'flight_id':[f'SIM-FS-{i:05d}' for i in range(1,N+1)],
 'route_group':route,
 'aircraft_id':aircraft_id,
 'scheduled_departure_hour':[f'{h:02d}:00' if i%9==0 else str(h) for i,h in enumerate(hour)],
 'planned_turnaround_min':[f'{x/60:.2f} h' if i%11==0 else (f'{x} min' if i%7==0 else str(x)) for i,x in enumerate(turnaround)],
 'inbound_delay_min':[('?' if i%61==0 else (f'{x/60:.2f} h' if i%13==0 else (f'{x} min' if i%5==0 else str(round(x,1))))) for i,x in enumerate(inbound)],
 'weather_disruption':[('?' if i%83==0 else ('adverse' if x=='Adverse' else ('clear' if i%4==0 else 'Clear'))) for i,x in enumerate(weather)],
 'demand_index':[('?' if i%97==0 else str(int(x))) for i,x in enumerate(demand)],
 'departure_delay_min':[f'{x/60:.2f} h' if i%17==0 and x>=60 else (f'{x} min' if i%6==0 else str(x)) for i,x in enumerate(dep_delay)],
 'delayed_15min_plus':['Yes' if x==0 else ('no' if i%4==0 else 'No') for i,x in enumerate(on_time)],
 'simulation_status':['SIMULATED']*N,
})
# Exact repeated rows are deliberately included as raw import duplicates; cleaning removes them.
raw=pd.concat([raw,raw.iloc[:12].copy()],ignore_index=True)
raw.to_csv('data_raw.csv',index=False,encoding='utf-8-sig')
print('wrote raw synthetic rows:',len(raw))


wrote raw synthetic rows: 1212


## 2. Clean the raw export

Cleaning removes exact duplicate record IDs, parses mixed but unambiguous date formats and hour strings, converts explicitly marked hours to minutes, normalizes category labels, and preserves missing numeric inputs as missing. The analysis excludes rows with missing inbound delay only from H0a; H0b has no missing test fields. All long delays are retained as operationally plausible values. See the data note for the bias paragraph and column dictionary.

In [2]:
import re, unicodedata
import numpy as np, pandas as pd
from pathlib import Path

def norm(v):
    if pd.isna(v): return ''
    return unicodedata.normalize('NFKC',str(v)).replace('\u00a0',' ').strip()

def minutes(v):
    s=norm(v).lower().replace(',','.')
    if s in {'','?','na','n/a','null','none','not reported'}: return np.nan
    m=re.search(r'[-+]?\d+(?:\.\d+)?',s)
    if not m: return np.nan
    x=float(m.group())
    if re.search(r'\bh\b|hour',s): x*=60
    return x

def clean_raw(raw):
    d=raw.copy()
    d['record_id']=pd.to_numeric(d['record_id'],errors='coerce').astype('Int64')
    d=d.drop_duplicates(subset=['record_id'],keep='first').copy()
    d['flight_date']=pd.to_datetime(d['flight_date'],format='mixed',dayfirst=True,errors='coerce').dt.date
    d['scheduled_departure_hour']=d['scheduled_departure_hour'].map(lambda v: int(str(v).split(':')[0]) if str(v).split(':')[0].isdigit() else np.nan)
    for c in ['planned_turnaround_min','inbound_delay_min','departure_delay_min']:
        d[c]=d[c].map(minutes)
    for c in ['route_group','aircraft_id','simulation_status']:
        d[c]=d[c].map(norm)
    d['weather_disruption']=d['weather_disruption'].map(norm).str.lower().map({'clear':'Clear','adverse':'Adverse','?':'Unknown','':'Unknown'}).fillna('Unknown')
    d['demand_index']=pd.to_numeric(d['demand_index'].replace({'?':np.nan,'' :np.nan}),errors='coerce')
    raw_flag=d['delayed_15min_plus'].map(norm).str.lower().map({'yes':1,'no':0})
    derived_flag=(d['departure_delay_min']>15).astype('int64')
    print('raw target label disagreements:',int((raw_flag!=derived_flag).sum()))
    d['delayed_15min_plus']=derived_flag
    d['record_id']=d['record_id'].astype('int64')
    d['scheduled_departure_hour']=d['scheduled_departure_hour'].astype('int64')
    d['demand_index']=d['demand_index'].astype('Int64')
    d['delayed_15min_plus']=d['delayed_15min_plus'].astype('int64')
    keep=['record_id','flight_date','flight_id','route_group','aircraft_id','scheduled_departure_hour','planned_turnaround_min','inbound_delay_min','weather_disruption','demand_index','departure_delay_min','delayed_15min_plus','simulation_status']
    d=d[keep].sort_values('record_id').reset_index(drop=True)
    return d

raw=pd.read_csv('data_raw.csv',dtype=str,keep_default_na=False)
clean=clean_raw(raw)
clean.to_csv('data_clean.csv',index=False,encoding='utf-8-sig')
print('raw rows',len(raw),'duplicate rows',len(raw)-raw.record_id.nunique(),'clean rows',len(clean))
print('target delay >15 min share',clean.delayed_15min_plus.mean())


raw target label disagreements: 0
raw rows 1212 duplicate rows 12 clean rows 1200
target delay >15 min share 0.16


## 3. Descriptive checks

In [3]:
df=pd.read_csv('data_clean.csv',parse_dates=['flight_date'])
print('Rows:',len(df),'| columns:',len(df.columns),'| duplicate record IDs:',df.record_id.duplicated().sum())
print('Date range:',df.flight_date.min().date(),'to',df.flight_date.max().date())
print('Departure delay mean/median/max (min):',df.departure_delay_min.mean(),df.departure_delay_min.median(),df.departure_delay_min.max())
print('Synthetic departure-delay proxy on-time share (<=15 min):',round((df.departure_delay_min<=15).mean(),4))
print('Missing values by column:\n',df.isna().sum().to_string())
print('Data status:',df.simulation_status.unique())

Rows: 1200 | columns: 13 | duplicate record IDs: 0
Date range: 2026-01-06 to 2026-12-08
Departure delay mean/median/max (min): 6.0265 0.0 55.8
Synthetic departure-delay proxy on-time share (<=15 min): 0.84
Missing values by column:
 record_id                    0
flight_date                  0
flight_id                    0
route_group                  0
aircraft_id                  0
scheduled_departure_hour     0
planned_turnaround_min       0
inbound_delay_min           20
weather_disruption           0
demand_index                13
departure_delay_min          0
delayed_15min_plus           0
simulation_status            0
Data status: <StringArray>
['SIMULATED']
Length: 1, dtype: str


## 4. Hypotheses and tests

**Pair A — inbound aircraft delay.** H0a: mean next-flight departure delay is equal for cases with inbound delay ≥15 minutes and <15 minutes. H1a: the mean is greater in the ≥15-minute group.

**Pair B — planned turnaround buffer.** H0b: mean departure delay is equal for planned turnarounds <45 minutes and ≥45 minutes. H1b: the mean is greater for the <45-minute group.

Both are one-sided Welch independent-samples t-tests, α=0.05. The data-generating process intentionally incorporates these assumed relationships; small p-values are expected in a simulation and are not empirical evidence about FlyArystan.

In [4]:
from scipy import stats
import numpy as np, pandas as pd

df=pd.read_csv('data_clean.csv',parse_dates=['flight_date'])
# H1a: inbound-arrival delay is associated with higher mean next-departure delay.
df['inbound_late_15']=df['inbound_delay_min']>=15
g1=df.loc[df.inbound_late_15,'departure_delay_min'].dropna()
g0=df.loc[~df.inbound_late_15,'departure_delay_min'].dropna()
r1=stats.ttest_ind(g1,g0,equal_var=False,alternative='greater')
# H1b: planned short turnaround buffer is associated with higher mean departure delay.
df['short_turnaround']=df['planned_turnaround_min']<45
g2=df.loc[df.short_turnaround,'departure_delay_min'].dropna()
g3=df.loc[~df.short_turnaround,'departure_delay_min'].dropna()
r2=stats.ttest_ind(g2,g3,equal_var=False,alternative='greater')

def welch_summary(a,b,res):
    diff=float(a.mean()-b.mean())
    se=np.sqrt(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b))
    dfree=(a.var(ddof=1)/len(a)+b.var(ddof=1)/len(b))**2/((a.var(ddof=1)/len(a))**2/(len(a)-1)+(b.var(ddof=1)/len(b))**2/(len(b)-1))
    tcrit=stats.t.ppf(.975,dfree)
    return {'n_group1':len(a),'n_group0':len(b),'mean_group1_min':float(a.mean()),'mean_group0_min':float(b.mean()),'difference_min':diff,'ci95_low':diff-tcrit*se,'ci95_high':diff+tcrit*se,'t_stat':float(res.statistic),'df_welch':float(dfree),'p_one_sided':float(res.pvalue)}

s1=welch_summary(g1,g0,r1); s2=welch_summary(g2,g3,r2)
print('Rows',len(df),'on-time proxy share',float((df.departure_delay_min<=15).mean()),'mean delay',float(df.departure_delay_min.mean()))
print('H0a: equal mean departure delay for inbound delay >=15 vs <15 min')
print(s1)
print('H0b: equal mean departure delay for planned turnaround <45 vs >=45 min')
print(s2)
print('Interpretation: p<0.05 rejects H0 only for this simulated sample, not for FlyArystan real operations.')


Rows 1200 on-time proxy share 0.84 mean delay 6.0265
H0a: equal mean departure delay for inbound delay >=15 vs <15 min
{'n_group1': 831, 'n_group0': 369, 'mean_group1_min': 7.95150421179302, 'mean_group0_min': 1.6913279132791328, 'difference_min': 6.260176298513888, 'ci95_low': np.float64(5.394317959482479), 'ci95_high': np.float64(7.126034637545296), 't_stat': 14.184966019128531, 'df_welch': 1193.9286321091615, 'p_one_sided': 1.2647605393476669e-42}
H0b: equal mean departure delay for planned turnaround <45 vs >=45 min
{'n_group1': 215, 'n_group0': 985, 'mean_group1_min': 11.55813953488372, 'mean_group0_min': 4.819086294416244, 'difference_min': 6.739053240467476, 'ci95_low': np.float64(5.065972968855865), 'ci95_high': np.float64(8.412133512079087), 't_stat': 7.930298187454973, 'df_welch': 268.8684297444973, 'p_one_sided': 2.943558707947817e-14}
Interpretation: p<0.05 rejects H0 only for this simulated sample, not for FlyArystan real operations.


## 5. Conclusion and limitations

Public evidence establishes that delays and passenger disruption occur, while also showing FlyArystan’s company-reported OTP improved to 84% in summer 2026 from 73% in summer 2025. The sources do not show that aircraft turnaround caused those delays; reported incidents include weather and a passenger/security event. The simulation only demonstrates how flight-level data could test two candidate explanations. Before operational action, obtain timestamped real flight/aircraft/turnaround records, include cancellations and cause codes, and repeat tests with aircraft/date clustering and controls for weather, route, airport, and schedule bank. Do not infer staffing or investment needs from these synthetic p-values.